# Herunterladen und Vorbereiten der Abhängigkeiten

In [ ]:
!pip install pillow

In [ ]:
!pip install matplotlib

In [ ]:
import requests
import random
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import time
import json

In [ ]:
Image.__version__ # '12.3.0'

# Datenvorbereitung

In [ ]:
GBIF_SPECIES_URL = "https://api.gbif.org/v1/species"
GBIF_OCCURRENCE_URL = "https://api.gbif.org/v1/occurrence/search"

def find_taxonomy(name):
    """Findet eine taxonomy anhand ihres Namens über die GBIF API"""

    params = {'name': name}

    response = requests.get(f"{GBIF_SPECIES_URL}/match", params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return None


In [ ]:
# Vögel 
bird_taxonomy = find_taxonomy("Aves")

# # Katzen
# cat_taxonomy = find_taxonomy("Felis catus")

# # Hunde
# dogs = find_taxonomy("Canis familiaris")

In [ ]:
print(bird_taxonomy)
# print(cat_taxonomy)
# print(dogs)

In [ ]:
def get_species(taxonomy_key):
    """Ruft Arten für einen angegebenen Taxonomie-Schlüssel über die GBIF API ab."""
    bird_species = {}
    limit = 200

    url = f"{GBIF_SPECIES_URL}/{taxonomy_key}/children"
    params = {
                'limit': limit,
                'highertaxon_key': taxonomy_key,
                'rank': 'SPECIES'
            }
    print(url)

    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()

        if data['results'] is not None:

            for species in data['results']:

                # print(f"Art: {species['scientificName']}, Schlüssel: {species['key']}")
        
                if species['key'] not in bird_species:
                    bird_species[species['key']] = {'scientificName': species['scientificName']}

                else:
                    continue

        return bird_species

    else:
        print(f"Error: {response.status_code}")
        return None

In [ ]:
birds_data = get_species(bird_taxonomy['usageKey'])
print(birds_data)
print(len(birds_data))

In [ ]:
def get_occurrences(species_key):

    params = {
        'speciesKey': species_key,
        'mediaType': 'StillImage',
        'occurrenceStatus': 'PRESENT',
        'limit': 5
    }

    response = requests.get(GBIF_OCCURRENCE_URL, params=params, timeout=50)
    
    if response.status_code == 200:
        time.sleep(.5)
        return response.json()
    elif response.status_code == 429:
        print(f"Warte 10 Sekunden.")
        time.sleep(10)
        return get_occurrences(species_key)
    elif response.status_code == 503:
        print(f"{response.status_code} Service Unavailable. Warte 10 Sekunden und versuche es erneut.")
        return None
    else:
        print(f'Error code: {response.status_code}')
        return None

In [ ]:
def is_valid_image(media, url):
    # print([media[i] for i in range(len(media))])
    text = "".join([media[i] for i in range(len(media))]).lower()

    unwanted_words = ['page', 'specimen', 'holotype', 'diagram', 'spectrogram', 'oscillogram', 'museum'] # um die Qualität der Bilder zu verbessern, werden bestimmte Schlüsselwörter in den Metadaten ausgeschlossen, die auf nicht-repräsentative Bilder hinweisen könnten.

    unwanted_url_parts = ['spectrogram', 'oscillogram']

    if any(word in text for word in unwanted_words):
        return False

    if any(part in url for part in unwanted_url_parts):
        return False

    # Prüfe bestimme domains, die bekannte nicht-representative Bilder wie Buchbilder oder nur Fossilbilder enthalten.
    if "images.collections.yale.edu" in url:
        return False

    return True


In [ ]:
def prepare_birds_data(birds_data):
    prepared_birds = {}
    i = 0
    l = len(birds_data)

    for species_key, species_data in birds_data.items():
        # print(f"Verarbeitete Art: {species_data['scientificName']} (Schlüssel für die Art: {species_key}). i = {i}")

        i += 1
        if i % 10 == 0:
            print(f"{'■'*int(round(i/10))} {'-'* int(round((l-i)/10))} -  {i}/{l} Arten verarbeitet.")

        occurences = get_occurrences(species_key)
        time.sleep(.3)

        if occurences is None:
            continue

        images = []
        image_urls = set()

        for occurence in occurences.get('results', []):

            for media in occurence.get('media', []):

                if media.get('type') != 'StillImage':
                    continue;

                image_url = media.get('identifier')

                if not is_valid_image([media.get('title', ''), media.get('description', ''), media.get('source', ''), (media.get('rightsHolder') or ''), media.get('identifier')], image_url):
                    continue                

                if image_url in image_urls:
                    continue;

                image_urls.add(image_url)
                images.append(media)

        if images:
            prepared_birds[species_key] = {
                'scientificName': species_data['scientificName'],
                'images': images
            }
   
    return prepared_birds

In [ ]:
def display_image(img_url):
    image_data = requests.get(img_url)
    image_data.raise_for_status()  # Löst eine Ausnahme aus, wenn die Anfrage unerfolgreich war.

    im = Image.open(BytesIO(image_data.content))

    plt.imshow(im)
    plt.axis('off')
    plt.show()

In [ ]:
cleaned_birds_data = prepare_birds_data(birds_data)

In [ ]:
print(len(cleaned_birds_data))
# print(cleaned_birds_data)
for i in cleaned_birds_data:
    print(cleaned_birds_data[i])

In [ ]:
def save_birds_data(cleaned_birds_data, filename):
    """Speichert die vorbereiteten Vogeldaten in einer JSON-Datei."""
    with open(filename, 'w') as f:
        json.dump(cleaned_birds_data, f, indent=4)

def load_birds_data(filename):
    """Ladet die vorbereiteten Vogeldaten aus einer JSON-DAtei."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print('Keine vorbereiteten Vogeldaten gefunden.')
        return None


In [ ]:
save_birds_data(cleaned_birds_data, 'cleaned_birds_data.json')

In [ ]:
cleaned_birds_data = load_birds_data('cleaned_birds_data.json')
len(cleaned_birds_data), cleaned_birds_data

# Spiel

In [ ]:
LEARNED_SPECIES = {}
MAX_NUMBER_OF_IMAGES = 3
LIVES = 3

In [ ]:
round(random.random()*3)

In [ ]:
cleaned_birds_data[random.choice(list(cleaned_birds_data.keys()))]

In [ ]:
def play_learning():
    while True:
        current_bird = random.choice(list(cleaned_birds_data.keys()))
        current_bird_images = cleaned_birds_data[current_bird]['images'][:MAX_NUMBER_OF_IMAGES]


        for i in range(len(current_bird_images)):
            display_image(current_bird_images[i]['identifier'])

            correct_answer = round(random.random() * 3)
            wrong_options = random.sample([species for species in cleaned_birds_data if species != current_bird], 3)
            options = wrong_options.copy()
            options.insert(correct_answer, current_bird)

            print('Welche der folgenden Antworten gehört zu diesem Bild?')
            # print(current_bird_images[i]['identifier'])

            for i, species_key in enumerate(options):
                print(f"{chr(65 + i)}-) {cleaned_birds_data[species_key]['scientificName']}")

            input_answer = input('Deine Antwort (A/B/C/D): ').strip().upper()

            if input_answer == chr(65 + correct_answer):
                print('Richtig!')
            else:
                print(f'Falsch! Die richtige Antwort war {cleaned_birds_data[current_bird]["scientificName"]}.')

        if current_bird not in LEARNED_SPECIES:
            LEARNED_SPECIES[current_bird] = cleaned_birds_data[current_bird]
            input_save = input('Möchtest du deinen Fortschritt speichern? (j/n): ').strip().lower()
            if input_save == 'j':
                save_state(LEARNED_SPECIES)
                print('Fortschritt gespeichert.')
            input_continue = input('Möchtest du weiterlernen? (j/n): ').strip().lower()
            if input_continue == 'n':
                print('Lernmodus wird beendet.')
                break
        else:
            input_continue = input('Möchtest du weiterlernen? (j/n): ').strip().lower()
            if input_continue == 'n':
                print('Lernmodus wird beendet.')
                break
            else:
                continue
                


In [ ]:
def play():
    score = 0
    learned_species = load_state().copy()
    print(len(learned_species))
    if len(learned_species) < 5:
        print("Sie müssen zuerst mindestens 5 verschiedene Vogelarten lernen, bevor Sie mit dem Test beginnen können.")
        return
    
    identified_species = set()
    lives = 3

    while lives > 0:
        print(len(identified_species))
        if len(learned_species) == 0:
            print(f"Glückwunsch! Sie haben alle gelernten Vogelarten erkannt. Ihre Punktzahl ist {score}")
            break

        print(f"Sie haben {'♥' * lives} Leben übrig.")

        current_bird = random.choice(list(learned_species.keys()))
        # print(f"Current bird: {learned_species[current_bird]['images']}") # Anzeige den Aktuellen Vogelart, um zu überprüfen, welche Bilder gültig sind.

        random_image = random.choice(learned_species[current_bird]['images'])['identifier']
        display_image(random_image)

        wrong_options = random.sample([species for species in cleaned_birds_data if species != current_bird], 3)
        options = wrong_options.copy()

        correct_answer = round(random.random() * 3)
        options.insert(correct_answer, current_bird)

        print('Welche den folgenden Antworten gehört zu diesem Bild? \n')
        print(learned_species[current_bird]['scientificName']) # Um zu überprüfen, ob der Code funktioniert

        choices = {'A': options[0], 'B': options[1], 'C': options[2], 'D': options[3]}
        correct_letter = chr(65 + correct_answer)

        for i, species_key in enumerate(options):
            print(f"{chr(65 + i)}-) {cleaned_birds_data[species_key]['scientificName']}")

        input_answer = input('Ihre Antwort (A/B/C/D): ').strip().upper()

        while input_answer not in ['A', 'B', 'C', 'D']:
            input_answer = input('Ungültige Eingabe. Bitte geben Sie A, B, C oder D ein: ').strip().upper()

        if choices[input_answer] == current_bird:
            score += 1
            identified_species.add(current_bird)
            learned_species.pop(current_bird)
            print('Richtig!')
        else:
            lives -= 1
            print(f'Falsch! Die richtige Antwort war {correct_letter}-) {learned_species[current_bird]["scientificName"]}. Sie haben {lives} Leben übrig.')


    if lives == 0:
        print("Spiel Vorbei! sie haben keine Leben mehr.")
        print(f"Ihr finaler Punktestand ist: {score}")

    
    input_continue = input('Möchten Sie weiter spielen? (j/n): ').strip().lower()
    while input_continue not in ['j', 'n']:
        input_continue = input('Ungültige Eingabe. Bitte geben Sie j oder n ein: ').strip().lower()
        
    if input_continue == 'j':
        print('Das Spiel wird fortgesetzt.')
        play()
    elif input_continue == 'n':
        print('Spiel wird beendet.')
        return score
    else:
        print('Ungültige Eingabe. Das Spiel wird beendet.')
        return score

    

In [ ]:
def save_state(LEARNED_SPECIES):
    """ Speichern der gelernten Vögel in eine Datei."""
    with open("learned_birds.txt", "w") as f:
        for species in LEARNED_SPECIES:
            f.write(f'{species} \n')

In [ ]:
def load_state():
    """ Laden der gelernten Vögel aus einer .txt-Datei."""
    learned_species = {}

    try:
        with open("learned_birds.txt", "r") as f:
            for text in f:
                species_key = text.strip()
                if species_key in cleaned_birds_data:
                    # print(cleaned_birds_data[species_key])

                    learned_species[species_key] = cleaned_birds_data[species_key]

            return learned_species
                    
    except FileNotFoundError:
        print("Kein gespeicherter Fortschritt gefunden.")
    

In [ ]:
print(LEARNED_SPECIES)
print(len(LEARNED_SPECIES))

In [ ]:
save_state(LEARNED_SPECIES)

In [ ]:
load_state()

In [ ]:
while True:
    choice = input("Geben Sie 'test', 'play', oder 'q' ein: ").strip().lower()

    if choice == "test":
        play_learning()
    elif choice == "play":
        play()
    elif choice == "q" or choice == "quit":
        break
    else:
        print("Ungültige Eingabe. Bitte geben Sie 'test', 'play', oder 'q / quit' ein.")

In [ ]:
# for i in cleaned_birds_data:
#     if cleaned_birds_data[i]['scientificName'] == 'Antiurus Ridgway, 1912':
#         print(cleaned_birds_data[i])
        